# Fine-tuning IndicBERT for Sanskrit POS Tagging (Improved)

**Model:** `ai4bharat/indic-bert` — a BERT-based encoder model trained on 12 Indic languages including Sanskrit.

**Task:** Word-level Part-of-Speech (POS) classification using morphological analysis data from the Bhagavata Purana JSON dataset.

**What each word is classified into:**
- `NOUN` — nouns (masculine / feminine / neuter). Note: Sanskrit adjectives also decline like nouns, so they appear here too.
- `VERB` — verbs (all classes 1–10)
- `PRON` — pronouns
- `PART` — particles and kṛdanta (participles, gerunds, infinitives)
- `IND` — indeclinables (covers adverbs, conjunctions, prepositions — Sanskrit has no separate ADV class)

**Improvements over v1:**
1. **Class-weighted loss** — penalises mistakes on rare classes (PRON, VERB) harder than NOUN
2. **Verse context input** — model sees the full verse alongside the word, not the word in isolation
3. **Label smoothing** — prevents overconfident wrong predictions (e.g. 98% confidence on wrong class)
4. **Macro F1 tracking** — honest metric that weighs all classes equally, not just the dominant NOUN class

**Before running:** Runtime → Change runtime type → **T4 GPU**

## Step 1 — Install dependencies

In [ ]:
!pip install transformers datasets accelerate indic-transliteration scikit-learn -q

## Step 2 — Upload the JSON dataset

**Instructions:**
1. On your computer, zip the entire `json/` folder → right-click → Send to → Compressed (zip) file. Name it `json_files.zip`.
2. Run the cell below, click **Choose Files**, and select `json_files.zip`.
3. Wait for the upload and extraction to finish.

In [ ]:
from google.colab import files
import zipfile, os

print("Upload your json_files.zip ...")
uploaded = files.upload()

os.makedirs('json_files', exist_ok=True)
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('json_files/')
        print(f"Extracted {fname}")

# Count extracted files
import glob
json_paths = glob.glob('json_files/**/*.json', recursive=True) + glob.glob('json_files/*.json')
json_paths = [p for p in json_paths if 'bhagavata_book' in os.path.basename(p)]
print(f"Found {len(json_paths)} chapter JSON files")

## Step 3a — Inspect raw lexical categories in your data

Before mapping categories to POS labels, we first print every unique `lexical_category`
string that actually exists in your JSON files. This tells us exactly what label mappings
we need — so we're not guessing.

Run this cell first, read the output, then proceed to Step 3b which does the actual loading.

In [ ]:
## Step 3b — Load and parse data

In [ ]:
import json, re, random
from collections import Counter
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# ── Verse cleaning ─────────────────────────────────────────────────────────────
_VERSE_NUM = re.compile(r'॥[\s\d०-९]+॥')

def clean_verse(text: str) -> str:
    text = _VERSE_NUM.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

def iast_to_devanagari(text: str) -> str:
    try:
        return transliterate(text, sanscript.IAST, sanscript.DEVANAGARI)
    except Exception:
        return text


# ── Broad POS mapping (used as the fallback) ───────────────────────────────────
def simplify_pos(lex_catg: str):
    lex = lex_catg.lower().strip()
    if 'pronoun' in lex:                                          return 'PRON'
    if 'noun' in lex:                                             return 'NOUN'
    if 'verb' in lex:                                             return 'VERB'
    if 'particle' in lex or 'krdanta' in lex or 'kṛdanta' in lex: return 'PART'
    if 'indecl' in lex or 'avyaya' in lex or 'prefix' in lex:     return 'IND'
    return None


# ── FIX #1: Label disambiguation ───────────────────────────────────────────────
# The JSON 'analyses' list is the morphological analyzer's CANDIDATE readings,
# returned noun-first. Blindly taking analyses[0] mislabels closed-class words
# (e.g. the pronoun 'eṣa' → "noun, neuter") and inflates NOUN to ~77%, teaching
# the model to just reproduce the analyzer's noun-first bias.
# We apply a few high-precision overrides BEFORE falling back to analyses[0].

# Closed-class stems — matched against each analysis' base_word_ and the surface
# form (IAST). Closed classes rarely collide with open-class words, so a lexicon
# hit is a reliable signal regardless of the analyzer's ordering.
PRON_WORDS = {
    'tad','etad','idam','adas','asmad','yuṣmad','kim','yad','eṣa','ena',
    'ayam','asau','ima','sa','sā','tat','etat','aham','tvam','mad','ka',
}
IND_WORDS = {
    'ca','vā','hi','eva','iva','api','tu','sma','uta','atha','atho',
    'vai','kila','khalu','nu','nūnam','iti','na','mā','yathā','tathā','evam',
    'yatra','tatra','kutra','yadā','tadā','kadā','sadā','punar','tāvat','yāvat',
    'adya','idānīm','nityam','satatam','purā','paścāt','saha','sārdham','vinā',
    'ṛte','prati','anu','antar','bahis','ūrdhvam','adhas','upari','agre',
}

def _norm(s: str) -> str:
    return s.lower().replace('√', '').strip(" '’‍").strip()

def _is_finite_verb(a: dict) -> bool:
    # Finite verbs are the only category marked for grammatical PERSON.
    lex   = a.get('lexical_category', '').lower()
    morph = a.get('morphological_and_syntactical_analysis', '').lower()
    return 'verb' in lex and any(p in morph for p in ('first', 'second', 'third'))

def _is_core_noun(a: dict) -> bool:
    # A noun in a core case (subject/object/address) is a plausible reading, so we
    # must NOT override it to VERB on the strength of a spurious finite-verb
    # candidate — e.g. 'rājo' (the king) also offers an imperative reading of √rāj.
    lex   = a.get('lexical_category', '').lower()
    morph = a.get('morphological_and_syntactical_analysis', '').lower()
    return 'noun' in lex and any(c in morph for c in ('nominative', 'accusative', 'vocative'))

def choose_pos(word_iast: str, analyses: list):
    surf  = _norm(word_iast)
    bases = {_norm(a.get('base_word_', '')) for a in analyses}
    forms = bases | {surf}

    # 1. Closed-class lexicon overrides (highest precision).
    if forms & PRON_WORDS:
        return 'PRON'
    if forms & IND_WORDS:
        return 'IND'

    # 2. Kṛdanta suffixes: gerund (-tvā) and infinitive (-tum) are unambiguous.
    if surf.endswith('tvā') or surf.endswith('tum'):
        return 'PART'

    # 3. Guarded finite-verb heuristic: prefer VERB only when a finite-verb
    #    reading exists AND no competing core-case noun reading does.
    if any(_is_finite_verb(a) for a in analyses) and not any(_is_core_noun(a) for a in analyses):
        return 'VERB'

    # 4. Fallback: trust the analyzer's first candidate.
    return simplify_pos(analyses[0].get('lexical_category', ''))


# ── Data extraction ────────────────────────────────────────────────────────────
examples      = []
skipped_files = 0

for path in sorted(json_paths):
    try:
        chapter_data = json.loads(open(path, encoding='utf-8').read())
    except Exception:
        skipped_files += 1
        continue

    for verse in chapter_data:
        verse_dev = clean_verse(verse.get('devanagari', '') or '')
        for item in verse.get('verse_Syn', []):
            if not isinstance(item, dict) or 'analyses' not in item:
                continue
            word_iast = item.get('word', '').strip()
            if not word_iast or not item['analyses']:
                continue
            pos_label = choose_pos(word_iast, item['analyses'])
            if pos_label is None:
                continue
            examples.append({
                'word'     : iast_to_devanagari(word_iast),
                'word_iast': word_iast,
                'verse'    : verse_dev,
                'label'    : pos_label,
            })

print(f"Total training examples : {len(examples):,}")
print(f"Files skipped (errors)  : {skipped_files}")
print()
print("Label distribution after disambiguation:")
for lbl, c in Counter(e['label'] for e in examples).most_common():
    print(f"  {lbl:<6} {c:>7,}  {c/len(examples)*100:5.1f}%")
print()
print("Sample entries (word → verse context → label):")
for ex in examples[:4]:
    print(f"  {ex['word_iast']:<20} | {ex['verse'][:55]}... | {ex['label']}")

In [ ]:
# NOTE: This is the same disambiguating loader as the cell above. It is kept here
# (identical) so the pipeline is correct regardless of which loader cell you run.
import json, re, random
from collections import Counter
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

_VERSE_NUM = re.compile(r'॥[\s\d०-९]+॥')

def clean_verse(text: str) -> str:
    text = _VERSE_NUM.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()

def iast_to_devanagari(text: str) -> str:
    try:
        return transliterate(text, sanscript.IAST, sanscript.DEVANAGARI)
    except Exception:
        return text


def simplify_pos(lex_catg: str):
    lex = lex_catg.lower().strip()
    if 'pronoun' in lex:                                          return 'PRON'
    if 'noun' in lex:                                             return 'NOUN'
    if 'verb' in lex:                                             return 'VERB'
    if 'particle' in lex or 'krdanta' in lex or 'kṛdanta' in lex: return 'PART'
    if 'indecl' in lex or 'avyaya' in lex or 'prefix' in lex:     return 'IND'
    return None


PRON_WORDS = {
    'tad','etad','idam','adas','asmad','yuṣmad','kim','yad','eṣa','ena',
    'ayam','asau','ima','sa','sā','tat','etat','aham','tvam','mad','ka',
}
IND_WORDS = {
    'ca','vā','hi','eva','iva','api','tu','sma','uta','atha','atho',
    'vai','kila','khalu','nu','nūnam','iti','na','mā','yathā','tathā','evam',
    'yatra','tatra','kutra','yadā','tadā','kadā','sadā','punar','tāvat','yāvat',
    'adya','idānīm','nityam','satatam','purā','paścāt','saha','sārdham','vinā',
    'ṛte','prati','anu','antar','bahis','ūrdhvam','adhas','upari','agre',
}

def _norm(s: str) -> str:
    return s.lower().replace('√', '').strip(" '’‍").strip()

def _is_finite_verb(a: dict) -> bool:
    lex   = a.get('lexical_category', '').lower()
    morph = a.get('morphological_and_syntactical_analysis', '').lower()
    return 'verb' in lex and any(p in morph for p in ('first', 'second', 'third'))

def _is_core_noun(a: dict) -> bool:
    lex   = a.get('lexical_category', '').lower()
    morph = a.get('morphological_and_syntactical_analysis', '').lower()
    return 'noun' in lex and any(c in morph for c in ('nominative', 'accusative', 'vocative'))

def choose_pos(word_iast: str, analyses: list):
    surf  = _norm(word_iast)
    bases = {_norm(a.get('base_word_', '')) for a in analyses}
    forms = bases | {surf}
    if forms & PRON_WORDS:
        return 'PRON'
    if forms & IND_WORDS:
        return 'IND'
    if surf.endswith('tvā') or surf.endswith('tum'):
        return 'PART'
    if any(_is_finite_verb(a) for a in analyses) and not any(_is_core_noun(a) for a in analyses):
        return 'VERB'
    return simplify_pos(analyses[0].get('lexical_category', ''))


examples      = []
skipped_files = 0

for path in sorted(json_paths):
    try:
        chapter_data = json.loads(open(path, encoding='utf-8').read())
    except Exception:
        skipped_files += 1
        continue

    for verse in chapter_data:
        verse_dev = clean_verse(verse.get('devanagari', '') or '')
        for item in verse.get('verse_Syn', []):
            if not isinstance(item, dict) or 'analyses' not in item:
                continue
            word_iast = item.get('word', '').strip()
            if not word_iast or not item['analyses']:
                continue
            pos_label = choose_pos(word_iast, item['analyses'])
            if pos_label is None:
                continue
            examples.append({
                'word'     : iast_to_devanagari(word_iast),
                'word_iast': word_iast,
                'verse'    : verse_dev,
                'label'    : pos_label,
            })

print(f"Total training examples : {len(examples):,}")
print(f"Files skipped (errors)  : {skipped_files}")
print()
print("Label distribution after disambiguation:")
for lbl, c in Counter(e['label'] for e in examples).most_common():
    print(f"  {lbl:<6} {c:>7,}  {c/len(examples)*100:5.1f}%")
print()
print("Sample entries:")
for ex in examples[:5]:
    print(f"  {ex['word_iast']:20s} → {ex['word']:20s} → {ex['label']}")

## Step 4 — Explore the label distribution

In [ ]:
label_counts = Counter(e['label'] for e in examples)
print("Label distribution:")
print(f"{'Label':<8} {'Count':>8} {'%':>7}")
print('-' * 28)
total = sum(label_counts.values())
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label:<8} {count:>8,} {count/total*100:>6.1f}%")

# Build label <-> id mappings
labels = sorted(label_counts.keys())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}
num_labels = len(labels)
print(f"\nNumber of classes: {num_labels}")
print(f"Label order: {labels}")

In [ ]:
## Step 5 — Train / validation split

import random

random.seed(42)
random.shuffle(examples)

split = int(0.9 * len(examples))
train_examples = examples[:split]
val_examples   = examples[split:]

print(f"Train examples : {len(train_examples):,}")
print(f"Val examples   : {len(val_examples):,}")

## Step 6 — Tokenise with IndicBERT tokenizer (with verse context)

**FIX 2 — Verse context:** Instead of feeding a word in isolation, we now feed a
*sentence pair*:

```
[CLS]  full verse in Devanagari  [SEP]  target word  [SEP]
```

IndicBERT's cross-attention can now see surrounding words when classifying each token.
This is critical for disambiguating VERB vs NOUN (e.g. `gacchati` in context clearly
behaves as a predicate, not a noun).

`truncation='only_first'` ensures the verse is shortened if needed, but the word
at the end is never cut off.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "ai4bharat/indic-bert"

# MAX_LEN increased from 32 → 128 to fit verse context + word.
# A Sanskrit verse is typically 30-60 Devanagari tokens; the word adds ~5.
# 128 tokens comfortably covers most verse+word pairs without excessive padding.
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Demo: show how a verse+word pair is tokenised into a sentence pair
sample = examples[0]
tokens = tokenizer.tokenize(
    sample['verse'][:60],  # abbreviated verse for display
    sample['word'],
)
print(f"\nSentence-pair tokenisation demo:")
print(f"  Verse (truncated) : {sample['verse'][:60]}")
print(f"  Word              : {sample['word']}")
print(f"  Tokens            : {tokens[:20]} ...")


def make_hf_dataset(example_list):
    # Build a HuggingFace Dataset from our list of dicts
    raw = Dataset.from_list([
        {'verse': e['verse'], 'word': e['word'], 'label': label2id[e['label']]}
        for e in example_list
    ])

    def tokenize_batch(batch):
        # FIX 2: Feed sentence A = verse (context), sentence B = word (target).
        # truncation='only_first' means: if total length > MAX_LEN, shorten
        # sentence A (the verse) but NEVER cut off sentence B (the word).
        return tokenizer(
            batch['verse'],          # sentence A — verse context
            batch['word'],           # sentence B — word to classify
            padding='max_length',
            truncation='only_first', # truncate verse if needed, never the word
            max_length=MAX_LEN,
        )

    tokenized = raw.map(
        tokenize_batch,
        batched=True,
        remove_columns=['verse', 'word'],
    )
    return tokenized


train_ds = make_hf_dataset(train_examples)
val_ds   = make_hf_dataset(val_examples)

print(f"\nTrain dataset : {train_ds}")
print(f"Val dataset   : {val_ds}")

## Step 6 — Tokenise with IndicBERT tokenizer

IndicBERT uses WordPiece tokenization. Each Sanskrit word (in Devanagari) is tokenized, padded to a fixed length, and the `[CLS]` token's representation is used for classification.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "ai4bharat/indic-bert"
MAX_LEN    = 32  # single Sanskrit words are short; 32 tokens is plenty

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size:,}")

# Quick demo of tokenization
sample_word = examples[0]['word']
tokens = tokenizer.tokenize(sample_word)
print(f"Word: {sample_word!r}  →  tokens: {tokens}")


def make_hf_dataset(example_list):
    raw = Dataset.from_list([
        {'word': e['word'], 'label': label2id[e['label']]}
        for e in example_list
    ])
    tokenized = raw.map(
        lambda batch: tokenizer(
            batch['word'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
        ),
        batched=True,
        remove_columns=['word'],
    )
    return tokenized


train_ds = make_hf_dataset(train_examples)
val_ds   = make_hf_dataset(val_examples)

print(f"\nTrain dataset: {train_ds}")
print(f"Val dataset  : {val_ds}")

## Step 8 — Define metrics

**FIX 4 — Macro F1:** We now track **both** weighted F1 and macro F1.

- `weighted_f1` — weighted by class size. Inflated by NOUN (77% of data). Used as secondary metric.
- `macro_f1` — weighs every class equally regardless of size. **This is the honest metric.** We use it to select the best checkpoint.

After training, a jump in `macro_f1` (e.g. 0.82 → 0.87) means minority classes like VERB and PRON genuinely improved — not just that the model got better at predicting the already-dominant NOUN.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

def compute_metrics(eval_pred):
    logits, label_ids = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(label_ids, preds)

    # weighted_f1: each class's F1 weighted by how many examples it has.
    # High because NOUN dominates. Good for checking overall performance.
    weighted_f1 = f1_score(label_ids, preds, average='weighted')

    # macro_f1: simple average of each class's F1, regardless of class size.
    # This is our primary metric — it penalises the model equally for missing
    # a rare PRON (144 examples) vs a common NOUN (13,984 examples).
    macro_f1 = f1_score(label_ids, preds, average='macro')

    return {
        'accuracy'   : acc,
        'macro_f1'   : macro_f1,
        'weighted_f1': weighted_f1,
    }

## Step 9 — Training with all four fixes applied

**FIX 1 — Class-weighted loss (`WeightedTrainer`):**  
Each class gets a weight inversely proportional to its frequency. PRON (144 examples)
gets ~97× more weight than NOUN (13,984 examples). The model is now penalised much
harder for misclassifying a rare PRON than for misclassifying a common NOUN.

**FIX 3 — Label smoothing (`label_smoothing_factor=0.1`):**  
Instead of training the model to output `[0, 0, 1, 0, 0]` for a certain VERB, we
train it towards `[0.02, 0.02, 0.92, 0.02, 0.02]`. This stops the model from becoming
overconfident (e.g. 98% confidence on the wrong class). The model will output more
honest, calibrated probabilities.

**FIX 4 — `metric_for_best_model='macro_f1'`:**  
The best checkpoint saved during training is now the one with the highest macro F1,
not the highest accuracy. This prevents saving a checkpoint that is great at NOUN
but terrible at VERB/PRON.

In [ ]:
import torch
from torch.nn import CrossEntropyLoss
from transformers import TrainingArguments, Trainer

# ── FIX 1: Compute inverse-frequency class weights ─────────────────────────────
# Formula: weight[i] = total_examples / (num_classes × count[i])
# This means: rare classes get high weight, common classes get low weight.
# Example with 5 classes and 18,000 examples:
#   NOUN  (13,984 examples) → weight ≈ 0.26  (penalised lightly)
#   PRON  (   144 examples) → weight ≈ 25.0  (penalised very heavily)
counts       = [label_counts[id2label[i]] for i in range(num_labels)]
total        = sum(counts)
class_weights = torch.tensor(
    [total / (num_labels * c) for c in counts],
    dtype=torch.float,
).to(device)

print("Class weights (higher = rarer class, penalised harder):")
for i, (lbl, w) in enumerate(zip(labels, class_weights.tolist())):
    print(f"  {lbl:<6}  count={counts[i]:>6,}  weight={w:.4f}")


# ── FIX 1: Custom Trainer that applies class weights to the loss ───────────────
# The standard HuggingFace Trainer uses unweighted CrossEntropyLoss.
# We override compute_loss to inject our class_weights tensor.
# Everything else (evaluation, saving, logging) stays exactly the same.
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Pop labels before forward pass (model doesn't expect them as input)
        labels  = inputs.pop("labels")
        outputs = model(**inputs)

        # CrossEntropyLoss with class_weights: rare-class errors cost more
        loss_fn = CrossEntropyLoss(weight=class_weights)
        loss    = loss_fn(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss


# ── Training arguments ─────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir              = './indicbert-sanskrit-pos-v2',
    num_train_epochs        = 5,        # more epochs now that loss is harder (weighted)
    per_device_train_batch_size = 32,   # reduced from 64 — sentence pairs are longer
    per_device_eval_batch_size  = 64,
    learning_rate           = 2e-5,     # standard BERT fine-tuning rate
    warmup_ratio            = 0.06,     # 6% of steps used for LR warm-up
    weight_decay            = 0.01,     # L2 regularisation to prevent overfitting

    # FIX 3: Label smoothing — prevents 99% confidence on wrong predictions.
    # 0.1 means: true label gets 0.9 probability mass, rest split across others.
    label_smoothing_factor  = 0.1,

    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    load_best_model_at_end  = True,

    # FIX 4: Use macro_f1 (not accuracy) to pick the best checkpoint.
    # Macro F1 is the honest metric — it weighs PRON and NOUN equally.
    metric_for_best_model   = 'macro_f1',
    greater_is_better       = True,

    logging_steps           = 100,
    report_to               = 'none',
    fp16                    = (device == 'cuda'),
)

# ── Create trainer (WeightedTrainer replaces plain Trainer) ───────────────────
trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
)

print("\nStarting training with all 4 fixes ...")
print("Watch 'macro_f1' column — that is the real accuracy across all classes.\n")
trainer.train()

## Step 9 — Training

Training for **3 epochs** on a T4 GPU takes roughly **5–15 minutes** depending on dataset size.

Key hyperparameters:
- `learning_rate=2e-5` — standard for fine-tuning BERT-like models
- `fp16=True` — half-precision training (faster on T4, uses less memory)
- `load_best_model_at_end=True` — keeps the checkpoint with the highest validation accuracy

In [ ]:
eval_results = trainer.evaluate()

print("=" * 55)
print("FINAL EVALUATION ON VALIDATION SET")
print("=" * 55)
print(f"\n  Accuracy     : {eval_results['eval_accuracy']:.4f}  ({eval_results['eval_accuracy']*100:.2f}%)")
print(f"  Macro F1     : {eval_results['eval_macro_f1']:.4f}  ← primary metric (all classes equal)")
print(f"  Weighted F1  : {eval_results['eval_weighted_f1']:.4f}  ← secondary (skewed by NOUN)")

# v1 baseline for comparison (from your previous run)
print()
print("  Improvement over v1:")
print(f"    Accuracy     : was 91.73%  → now {eval_results['eval_accuracy']*100:.2f}%")
print(f"    Macro F1     : was  0.8184 → now {eval_results['eval_macro_f1']:.4f}")
print(f"    (v1 Macro F1 was the real bottleneck — watch this number improve)")

# Per-class breakdown — this shows whether the fixes actually helped PRON and VERB
print("\n" + "=" * 55)
print("PER-CLASS BREAKDOWN")
print("=" * 55)
print("(Compare PRON and VERB F1 with v1 baseline: PRON was 0.7394, VERB was 0.7932)\n")

raw_preds = trainer.predict(val_ds)
preds     = np.argmax(raw_preds.predictions, axis=-1)
true_ids  = raw_preds.label_ids

report = classification_report(true_ids, preds, target_names=labels, digits=4)
print(report)

## Step 10 — Evaluate: accuracy and per-class F1

In [ ]:
print("=" * 50)
print("FINAL EVALUATION ON VALIDATION SET")
print("=" * 50)

eval_results = trainer.evaluate()
print(f"\n  Accuracy    : {eval_results['eval_accuracy']:.4f}  ({eval_results['eval_accuracy']*100:.2f}%)")
print(f"  Weighted F1 : {eval_results['eval_weighted_f1']:.4f}")

# Per-class breakdown
print("\n" + "=" * 50)
print("PER-CLASS BREAKDOWN")
print("=" * 50)

# Run prediction on validation set
raw_preds = trainer.predict(val_ds)
preds     = np.argmax(raw_preds.predictions, axis=-1)
true_ids  = raw_preds.label_ids

report = classification_report(
    true_ids, preds,
    target_names=labels,
    digits=4
)
print(report)

## Step 12 — Inference on custom Sanskrit words

`predict_pos` now accepts an optional `verse_iast` context string.
- **With context** → more accurate (matches how the model was trained)
- **Without context** → still works, falls back to word-only mode

We show both modes on the same test words so you can directly compare.

In [ ]:
model.eval()

def predict_pos(word_iast: str, verse_iast: str = '') -> tuple:
    """
    Predict POS for a Sanskrit word in IAST transliteration.

    Args:
        word_iast  : The word to classify (IAST), e.g. 'gacchati'
        verse_iast : Optional — the verse the word appears in (IAST).
                     Providing this matches training conditions and improves accuracy.

    Returns:
        (predicted_label, confidence_percent, devanagari_word)
    """
    word_dev = iast_to_devanagari(word_iast)

    if verse_iast.strip():
        # WITH context: [CLS] verse [SEP] word [SEP]  ← matches training
        verse_dev = iast_to_devanagari(verse_iast)
        inputs = tokenizer(
            verse_dev, word_dev,
            return_tensors='pt',
            max_length=MAX_LEN,
            padding='max_length',
            truncation='only_first',
        )
    else:
        # WITHOUT context: [CLS] word [SEP]  ← fallback, less accurate
        inputs = tokenizer(
            word_dev,
            return_tensors='pt',
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
        )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits

    # softmax converts raw logit scores to probabilities that sum to 1
    probs      = torch.softmax(logits, dim=-1)[0]
    pred_id    = probs.argmax().item()
    pred_label = id2label[pred_id]
    confidence = probs[pred_id].item()
    return pred_label, confidence, word_dev


# ── Test words with their actual verse contexts from the dataset ───────────────
# We pick real examples so the verse context is genuinely informative.
test_cases = [
    # (word_iast,   verse_iast from a real verse,                             expected)
    ('dharma',    '',                                                          'NOUN'),
    ('gacchati',  'tatra tatrāñjasā āyuṣman bhavatā yad viniścitam',         'VERB'),
    ('uvāca',     'śrīśuka uvāca varīyān eṣa te praśnaḥ kṛto lokahitaṃ nṛpa','VERB'),
    ('eṣa',       'śrīśuka uvāca varīyān eṣa te praśnaḥ kṛto lokahitaṃ nṛpa','PRON'),
    ('ca',        'anvayāt itarataḥ ca artheṣu abhijñaḥ svarāṭ',            'IND'),
    ('kṛṣṇa',    '',                                                          'NOUN'),
]

print("─" * 72)
print(f"{'Word':<14} {'Predicted':>10}  {'Conf':>7}  {'Expected':>10}  {'✓?':>4}")
print("─" * 72)
for word_iast, verse_iast, expected in test_cases:
    label, conf, dev = predict_pos(word_iast, verse_iast)
    correct = "✓" if label == expected else "✗"
    ctx     = "(with context)" if verse_iast else "(no context) "
    print(f"{word_iast:<14} {label:>10}  {conf:>6.1%}  {expected:>10}  {correct}  {ctx}")

# ── Show the confidence distribution for one word (calibration check) ──────────
print("\nConfidence breakdown for 'gacchati' (in context):")
word_iast   = 'gacchati'
verse_iast  = 'tatra tatrāñjasā āyuṣman bhavatā yad viniścitam'
word_dev    = iast_to_devanagari(word_iast)
verse_dev   = iast_to_devanagari(verse_iast)
inputs      = tokenizer(verse_dev, word_dev, return_tensors='pt',
                        max_length=MAX_LEN, padding='max_length', truncation='only_first')
inputs      = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    logits  = model(**inputs).logits
probs       = torch.softmax(logits, dim=-1)[0]
for i, lbl in enumerate(labels):
    bar = '█' * int(probs[i].item() * 40)
    print(f"  {lbl:<6} {probs[i].item():>6.1%}  {bar}")

## Step 12 — Inference on custom Sanskrit words

Enter any Sanskrit word in IAST transliteration to see the model's POS prediction.

In [ ]:
model.eval()

def predict_pos(word_iast: str) -> str:
    """Predict POS for a single Sanskrit word given in IAST transliteration."""
    word_dev = iast_to_devanagari(word_iast)
    inputs   = tokenizer(word_dev, return_tensors='pt', max_length=MAX_LEN,
                         padding='max_length', truncation=True)
    inputs   = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs      = torch.softmax(logits, dim=-1)[0]
    pred_id    = probs.argmax().item()
    pred_label = id2label[pred_id]
    confidence = probs[pred_id].item()
    return pred_label, confidence, word_dev


test_words = [
    'dharma',       # expected: NOUN
    'jñāna',        # expected: NOUN
    'gacchati',     # expected: VERB (goes)
    'mahān',        # expected: ADJ (great)
    'ca',           # expected: IND or PART (and)
    'eṣa',          # expected: PRON (this)
    'sadā',         # expected: ADV (always)
    'kṛṣṇa',        # expected: NOUN
]

print(f"{'IAST Word':<15} {'Devanagari':<18} {'Predicted POS':<14} {'Confidence'}")
print('-' * 65)
for w in test_words:
    label, conf, dev = predict_pos(w)
    print(f"{w:<15} {dev:<18} {label:<14} {conf:.2%}")

## Step 13 — Save the fine-tuned model

Downloads the model as a zip to your local machine.

In [ ]:
import shutil

SAVE_DIR = 'indicbert-sanskrit-pos-final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label mappings for later use
with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id}, f, indent=2)

shutil.make_archive('indicbert_sanskrit_pos', 'zip', SAVE_DIR)
files.download('indicbert_sanskrit_pos.zip')
print("Model saved and download started.")